# Metabolomics pipeline

Simulate a metabolomics matrix (3 groups, many features), run PCA, a top-variable heatmap and a fold-change / significance volcano with multiple-testing control.

In [ ]:
import numpy as np
from scipy import stats
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
rng = np.random.default_rng(21)
n_feat, n_per = 300, 15
groups = np.repeat([0, 1, 2], n_per)
base = np.exp(rng.normal(0, 1, n_feat))[:, None]
X = rng.gamma(shape=base*3, scale=1/3, size=(n_feat, n_per*3))
for g in (1, 2):
    idx = rng.choice(n_feat, 60, replace=False)
    X[np.ix_(idx, np.where(groups == g)[0])] *= rng.uniform(2, 5, size=(len(idx), 1))

In [ ]:
logX = np.log1p(X / X.mean(axis=1, keepdims=True) * 1e3)
pc = PCA(n_components=3, random_state=0).fit_transform(logX.T)
fig, ax = plt.subplots(figsize=(6, 5))
for g in range(3):
    ax.scatter(pc[groups == g, 0], pc[groups == g, 1], label=f'group {g}', s=25)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend()
ax.set_title('PCA score plot')

In [ ]:
order = np.argsort(logX.std(axis=1))[::-1][:30]
fig, ax = plt.subplots(figsize=(6, 7))
im = ax.imshow(logX[order], aspect='auto', cmap='viridis')
fig.colorbar(im, label='log intensity')
ax.set_ylabel('feature'); ax.set_xlabel('sample')
ax.set_title('Top-variable features')

In [ ]:
def bh(p):
    p = np.asarray(p); order = np.argsort(p)
    adj = np.empty_like(p); adj[order] = p[order]*len(p)/np.arange(1, len(p)+1)
    return np.minimum.accumulate(adj[::-1])[::-1]
ctrl = X[:, groups == 0]; trt = X[:, groups == 1]
logfc = np.log2((trt.mean(axis=1)+1e-6)/(ctrl.mean(axis=1)+1e-6))
pv = np.array([stats.ttest_ind(ctrl[g], trt[g], equal_var=False).pvalue for g in range(n_feat)])
q = bh(pv)
sig = q < 0.05
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(logfc, -np.log10(q+1e-12), s=4, alpha=0.6)
ax.scatter(logfc[sig], -np.log10(q[sig]+1e-12), s=8, color='#e05b5b')
ax.set_xlabel('log2 FC (group1 vs 0)'); ax.set_ylabel('-log10 q')
ax.set_title(f'Volcano — {sig.sum()} features q<0.05')